In [2]:
#Install required libraries
!pip install -q transformers==4.46.3
!pip install -q datasets==3.2.0
!pip install -q evaluate==0.4.3
!pip install -q rouge_score
!pip install -q sentencepiece
!pip install -q accelerate
!pip install -q flask

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  P

In [3]:
#Verify installations
import torch
import transformers
import datasets
import evaluate
import flask
import sentencepiece
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("GPU Available:", torch.cuda.is_available())
print("\nAll required libraries imported successfully!")

PyTorch: 2.14.1+cu130
Transformers: 4.46.3
Datasets: 3.2.0
Evaluate: 0.4.3
GPU Available: True

All required libraries imported successfully!


In [4]:
#Import libraries
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate
import os
import warnings
warnings.filterwarnings("ignore")
print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
#Check GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available!")
    print("GPU Name:",
          torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU not available. Using CPU.")
print("Selected device:", device)

GPU is available!
GPU Name: Tesla T4
Selected device: cuda


In [6]:
#Download the actual CNN/DailyMail dataset to train T5
#Downloading the original CNN/DailyMail version 3.0.0 dataset from Hugging Face.
dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0"
)
print(dataset)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [7]:
#Inspect the actual dataset structure
#Checking the actual columns before writing preprocessing code.
print("Dataset splits:")
print(dataset.keys())
print("\nTraining dataset size:")
print(len(dataset["train"]))
print("\nValidation dataset size:")
print(len(dataset["validation"]))
print("\nTesting dataset size:")
print(len(dataset["test"]))
print("\nActual columns:")
print(dataset["train"].column_names)
print("\nFirst record:")
print(dataset["train"][0])

Dataset splits:
dict_keys(['train', 'validation', 'test'])

Training dataset size:
287113

Validation dataset size:
13368

Testing dataset size:
11490

Actual columns:
['article', 'highlights', 'id']

First record:
{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds --

In [8]:
#Understand and display the dataset
# Identify columns from the actual dataset
columns = dataset["train"].column_names
print("Available columns:", columns)
# Check that required columns exist
assert "article" in columns, "Article column missing!"
assert "highlights" in columns, "Highlights column missing!"
#article is our input.
ARTICLE_COLUMN = "article"
#highlights is our expected output.
SUMMARY_COLUMN = "highlights"
#One record contains one article and its corresponding summary.
print("\nInput column:", ARTICLE_COLUMN)
print("Target column:", SUMMARY_COLUMN)
# Display one article-summary pair
sample = dataset["train"][0]
print("\nORIGINAL ARTICLE:\n")
print(sample[ARTICLE_COLUMN][:1500])
#T5 learns to generate the highlights from the article.
print("\nORIGINAL SUMMARY:\n")
print(sample[SUMMARY_COLUMN])

Available columns: ['article', 'highlights', 'id']

Input column: article
Target column: highlights

ORIGINAL ARTICLE:

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the h

In [9]:
#Select a manageable dataset subset
#Selecting a smaller portion of the original dataset for practical training.
TRAIN_SIZE = 2000
VALIDATION_SIZE = 300
TEST_SIZE = 300
# Shuffle training records and select a subset
train_dataset = dataset["train"].shuffle(
    seed=42
).select(range(TRAIN_SIZE))
# Select validation records
validation_dataset = dataset["validation"].shuffle(
    seed=42
).select(range(VALIDATION_SIZE))
# Select test records
test_dataset = dataset["test"].shuffle(
    seed=42
).select(range(TEST_SIZE))
print("Training records:", len(train_dataset))
print("Validation records:", len(validation_dataset))
print("Testing records:", len(test_dataset))

Training records: 2000
Validation records: 300
Testing records: 300


In [10]:
#Basic preprocessing the data
#We do not need complicated cleaning because CNN/DailyMail already contains prepared news articles and summaries.
def preprocess_text(example):
    article = example["article"].strip()
#strip() removes unnecessary spaces at the beginning and end.
    summary = example["highlights"].strip()
    return {
        "article": article,
        "highlights": summary
    }
#We preserve the original article content.We preserve the original reference summary.No artificial records are created.
train_dataset = train_dataset.map(
    preprocess_text
)
validation_dataset = validation_dataset.map(
    preprocess_text
)
test_dataset = test_dataset.map(
    preprocess_text
)
print("Preprocessing completed!")
print("\nProcessed article:")
print(train_dataset[0]["article"][:500])
print("\nProcessed summary:")
print(train_dataset[0]["highlights"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Preprocessing completed!

Processed article:
By . Anthony Bond . PUBLISHED: . 07:03 EST, 2 March 2013 . | . UPDATED: . 08:07 EST, 2 March 2013 . Three members of the same family who died in a static caravan from carbon monoxide poisoning would have been unconscious 'within minutes', investigators said today. The bodies of married couple John and Audrey Cook were discovered alongside their daughter, Maureen, at the mobile home they shared on Tremarle Home Park in Camborne, west Cornwall. The inquests have now opened into the deaths last Sat

Processed summary:
John and .
Audrey Cook were discovered alongside their daughter, Maureen .
They were found at Tremarle Home Park in Cornwall .
Investigators say the three died of carbon monoxide .
poisoning .


In [11]:
#Load the T5 tokenizer
MODEL_NAME = "google-t5/t5-small"

tokenizer = T5Tokenizer.from_pretrained(
    MODEL_NAME
)
print("T5 tokenizer loaded successfully!")
print("Vocabulary size:")
print(tokenizer.vocab_size)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5 tokenizer loaded successfully!
Vocabulary size:
32000


In [12]:
#Tokenize the article and summary
#Converting both articles and summaries into numerical token sequences.
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
def tokenize_data(batch):
    # Add T5 task prefix to the article
    #summarize: tells T5 which task to perform.
    inputs = [
        "summarize: " + article
        for article in batch["article"]
    ]
    # Tokenize input articles
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )
    # Tokenize target summaries
    labels = tokenizer(
        text_target=batch["highlights"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )
    #input_ids contains numerical representations of the article.
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
tokenized_train = train_dataset.map(
    tokenize_data,
    batched=True,
    remove_columns=train_dataset.column_names
)
tokenized_validation = validation_dataset.map(
    tokenize_data,
    batched=True,
    remove_columns=validation_dataset.column_names
)
#attention_mask identifies valid input positions.
#labels contains the numerical representation of the target summary.
tokenized_test = test_dataset.map(
    tokenize_data,
    batched=True,
    remove_columns=test_dataset.column_names
)
print("Tokenization completed!")
print(tokenized_train[0].keys())

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenization completed!
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [13]:
#Understand the tokenized dataset
example = tokenized_train[0]
#Input IDs represent the article.
print("Input IDs:")
print(example["input_ids"][:30])
#Attention masks identify valid tokens.
print("\nAttention Mask:")
print(example["attention_mask"][:30])
#Target IDs represent the expected summary.
print("\nTarget IDs:")
print(example["labels"][:30])
print("\nInput token count:")
print(len(example["input_ids"]))
print("\nTarget token count:")
print(len(example["labels"]))

Input IDs:
[21603, 10, 938, 3, 5, 11016, 12528, 3, 5, 3, 10744, 8775, 20619, 2326, 10, 3, 5, 10668, 10, 4928, 3, 6038, 6, 204, 1332, 2038, 3, 5, 1820, 3]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Target IDs:
[1079, 11, 3, 5, 31423, 6176, 130, 3883, 5815, 70, 3062, 6, 7758, 60, 35, 3, 5, 328, 130, 435, 44, 4691, 1635, 109, 1210, 1061, 16, 26334, 3, 5]

Input token count:
512

Target token count:
48


In [14]:
#Load the pretrained T5 model and Loading Google's pretrained T5-small model.
#Training a Transformer from scratch would require huge amounts of data and computing resources.
#Instead, we start with a model that has already learned language patterns.
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME
)
model = model.to(device)
print("Pretrained T5 model loaded successfully!")
#Pretrained means the model already has learned parameters.
print("Model:", MODEL_NAME)
#Fine-tuning means teaching those parameters about our specific task.We fine-tune T5 using CNN/DailyMail articles and summaries
print("Device:", device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Pretrained T5 model loaded successfully!
Model: google-t5/t5-small
Device: cuda


In [15]:
#Prepare the T5 dataset for training
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    ##Padding will be added dynamically when batches are prepared.
    padding=True
)
print("Data collator prepared!")
#The collator prepares batches.It dynamically pads sequences.It handles padding in target labels correctly.We don't need to manually add padding to every article.

Data collator prepared!


In [16]:
#Configure fine-tuning
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/t5_summarization_results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=1,
    load_best_model_at_end=False,
    seed=42
)
print("Training configuration completed!")

Training configuration completed!


In [17]:
#Fine-tune T5
#Training T5 using the real CNN/DailyMail article-summary pairs.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator
)
print("Starting T5 fine-tuning...")
trainer.train()
print("\nT5 training completed!")

Starting T5 fine-tuning...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,2.107300,1.808050



T5 training completed!


In [18]:
#Generate a summary Using our fine-tuned model to generate a summary.
def generate_summary(article):
    model.eval()
    input_text = "summarize: " + article
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    ).to(device)
    with torch.no_grad():
      #model.generate() generates the summary.
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_TARGET_LENGTH,
            min_length=20,
            #num_beams=4 allows the model to explore multiple possible output sequences.
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
        #tokenizer.decode() converts generated token IDs into readable text.
    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )
    return summary

In [19]:
#Test using real test dataset articles
for i in range(3):

    article = test_dataset[i]["article"]
    original_summary = test_dataset[i]["highlights"]
    generated_summary = generate_summary(article)
    print("\n" + "=" * 70)
    print("TEST ARTICLE:", i + 1)
    print("\nORIGINAL ARTICLE:\n")
    print(article[:1200])
    print("\nORIGINAL SUMMARY:\n")
    print(original_summary)
    print("\nGENERATED SUMMARY:\n")
    print(generated_summary)
    print("=" * 70)


TEST ARTICLE: 1

ORIGINAL ARTICLE:

(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I see it in the new surgeon general who cites data showing just how helpful it can be. I see a revolution in the attitudes of everyday Americans. For the first time a majority, 53%, favor its legalization, with 77% supporting it for medical purposes. Support for legalization has risen 11 points in the past few years alone. In 1969, the first time Pew asked the question about legalization, only 12% of the nation was in favor. I see a revolution that is burning white hot among young people, but also shows up am

In [20]:
#Evaluate using ROUGE
#Measuring the similarity between generated summaries and reference summaries.
#ROUGE stands for Recall-Oriented Understudy for Gisting Evaluation.
#It measures the overlap between generated summaries and reference summaries.
rouge = evaluate.load("rouge")
predictions = []
references = []
# Evaluate on the held-out test subset
for i in range(len(test_dataset)):
    article = test_dataset[i]["article"]
    reference = test_dataset[i]["highlights"]
    prediction = generate_summary(article)
    predictions.append(prediction)
    references.append(reference)
rouge_results = rouge.compute(
    predictions=predictions,
    references=references
)
print("ROUGE Evaluation Results:\n")
for metric, score in rouge_results.items():
    print(metric, ":", round(score, 4))

ROUGE Evaluation Results:

rouge1 : 0.3991
rouge2 : 0.1857
rougeL : 0.2819
rougeLsum : 0.343


In [21]:
#Test with your own article
print("AUTOMATIC TEXT SUMMARIZATION SYSTEM")

article_text = input(
    "\nEnter your article:\n"
)
if article_text.strip():
    result = generate_summary(article_text)
    print("\nGENERATED SUMMARY:\n")
    print(result)
else:
    print("Please enter some text.")

AUTOMATIC TEXT SUMMARIZATION SYSTEM

Enter your article:
Digital payment systems are changing the way small businesses manage their daily transactions. Shop owners, restaurants and local service providers are increasingly accepting payments through mobile applications and QR codes. These systems reduce the need to handle physical cash and make transactions more convenient for customers. Business owners can also maintain digital transaction records and monitor their daily sales. Financial experts believe that digital payments can help small businesses become more organized and improve financial transparency. However, businesses must remain careful about online fraud, payment security and technical failures. Customer awareness and secure payment systems are important for building trust.

GENERATED SUMMARY:

Shop owners, restaurants and local service providers are increasingly accepting payments through mobile applications and QR codes. These systems reduce the need to handle physical cas

In [22]:
#Save the trained model and tokenizer
SAVE_PATH = "/content/t5_summarization_model"

# Save trained model

model.save_pretrained(SAVE_PATH)

# Save tokenizer

tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

print("Saved location:", SAVE_PATH)

Model saved successfully!
Saved location: /content/t5_summarization_model


In [23]:
import shutil

shutil.make_archive(
    "/content/t5_summarization_model",
    "zip",
    SAVE_PATH
)

from google.colab import files

files.download(
    "/content/t5_summarization_model.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
#Create the Flask application
from flask import Flask, request, jsonify
app = Flask(__name__)
@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "T5 Automatic Text Summarization API is running!"
    })
@app.route("/summarize", methods=["POST"])
def summarize():
    try:
        data = request.get_json()
        if not data or "article" not in data:
            return jsonify({
                "error": "Please provide article text."
            }), 400
        article = data["article"]
        if not isinstance(article, str) or not article.strip():
            return jsonify({
                "error": "Article must be a non-empty string."
            }), 400
        # Generate summary using fine-tuned T5
        summary = generate_summary(article)
        return jsonify({
            "summary": summary
        })
    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500
print("Flask application created successfully!")

Flask application created successfully!


In [25]:
#Test the Flask API
# Create Flask test client
client = app.test_client()
# Check home endpoint
home_response = client.get("/")
print("HOME ENDPOINT:")
print(home_response.json)
# Prepare article for testing
test_article = """
The government has announced a new education policy
to improve the quality of education across the country.
The policy focuses on digital learning, teacher training,
modern classrooms and improved educational facilities.
The government expects this initiative to benefit students
and improve learning outcomes in schools and colleges.
"""
# Send POST request
response = client.post(
    "/summarize",
    json={
        "article": test_article
    }
)
print("\nSUMMARIZATION API RESPONSE:")
print(response.status_code)
print(response.json)

HOME ENDPOINT:
{'message': 'T5 Automatic Text Summarization API is running!'}

SUMMARIZATION API RESPONSE:
200
{'summary': 'The policy focuses on digital learning, teacher training, modern classrooms and improved educational facilities. The government expects this initiative to benefit students and improve learning outcomes in schools and colleges.'}
